In [ ]:
# install dependencies for local run
!pip install geopandas openpyxl matplotlib

In [ ]:
# import libraries
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
import os
# Set working directory to the Emily folder where the data files live
os.chdir(os.path.dirname(os.path.abspath("__file__")))
print("Working directory:", os.getcwd())

In [ ]:
# load data — update paths to match your environment
# Google Colab: use /content/myGdrive/MyDrive/ABT 182/project/
# Local: use relative paths like "Emily/Emily/"
df2014 = pd.read_csv("2014.csv")
df2016 = pd.read_csv("2016.csv")
df2018 = pd.read_csv("2018 - 2018.csv")
df2019 = pd.read_csv("2019 - 2019.csv")
df2020 = pd.read_csv("2020 - 2020.csv")
df2021 = pd.read_csv("2021.csv")
df2022 = pd.read_csv("2022.csv")

# Filtering ALL grapes + Calculating aggregate yield (tons/acre) by county
# Uses total Production (tons) for all grape varieties per county,
# divided by vineyard acreage per county from DWR data.

In [ ]:
# filter ALL grapes (wine, raisin, table, unspecified) — exclude grapefruit
# also exclude "State Total" and "Sum of Others" rows
grapes2014 = df2014[
    df2014["Crop Name"].str.contains("GRAPES", case=False, na=False)
    & ~df2014["Crop Name"].str.contains("GRAPEFRUIT", case=False, na=False)
    & ~df2014["County"].isin(["State Total", "State Totals", "Sum of Others"])
]

# Sum total production (tons) by county across all grape varieties
county_prod2014 = grapes2014.groupby("County")["Production"].sum().reset_index()
county_prod2014.columns = ["county", "total_production_tons"]

print(f"Grape varieties included: {sorted(grapes2014['Crop Name'].unique())}")
print(county_prod2014.head(10))

In [ ]:
# filter ALL grapes 2016
grapes2016 = df2016[
    df2016["Crop Name"].str.contains("GRAPES", case=False, na=False)
    & ~df2016["Crop Name"].str.contains("GRAPEFRUIT", case=False, na=False)
    & ~df2016["County"].isin(["State Total", "State Totals", "Sum of Others"])
]

county_prod2016 = grapes2016.groupby("County")["Production"].sum().reset_index()
county_prod2016.columns = ["county", "total_production_tons"]

print(f"Grape varieties included: {sorted(grapes2016['Crop Name'].unique())}")
print(county_prod2016.head(10))

In [ ]:
# filter ALL grapes 2018
grapes2018 = df2018[
    df2018["Crop Name"].str.contains("GRAPES", case=False, na=False)
    & ~df2018["Crop Name"].str.contains("GRAPEFRUIT", case=False, na=False)
    & ~df2018["County"].isin(["State Total", "State Totals", "Sum of Others"])
]

county_prod2018 = grapes2018.groupby("County")["Production"].sum().reset_index()
county_prod2018.columns = ["county", "total_production_tons"]

print(f"Grape varieties included: {sorted(grapes2018['Crop Name'].unique())}")
print(county_prod2018.head(10))

In [ ]:
# filter ALL grapes 2019
grapes2019 = df2019[
    df2019["Crop Name"].str.contains("GRAPES", case=False, na=False)
    & ~df2019["Crop Name"].str.contains("GRAPEFRUIT", case=False, na=False)
    & ~df2019["County"].isin(["State Total", "State Totals", "Sum of Others"])
]

county_prod2019 = grapes2019.groupby("County")["Production"].sum().reset_index()
county_prod2019.columns = ["county", "total_production_tons"]

print(f"Grape varieties included: {sorted(grapes2019['Crop Name'].unique())}")
print(county_prod2019.head(10))

In [ ]:
# filter ALL grapes 2020
grapes2020 = df2020[
    df2020["Crop Name"].str.contains("GRAPES", case=False, na=False)
    & ~df2020["Crop Name"].str.contains("GRAPEFRUIT", case=False, na=False)
    & ~df2020["County"].isin(["State Total", "State Totals", "Sum of Others"])
]

county_prod2020 = grapes2020.groupby("County")["Production"].sum().reset_index()
county_prod2020.columns = ["county", "total_production_tons"]

print(f"Grape varieties included: {sorted(grapes2020['Crop Name'].unique())}")
print(county_prod2020.head(10))

In [ ]:
# 2021: different column name ("Current Item Name")
# Use "Grapes, All" where available; for counties with only sub-type aggregates
# (e.g. "Grapes, Wine, All"), sum those instead to avoid missing data.
all_grapes2021 = df2021[
    df2021["Current Item Name"].str.contains("Grapes", case=False, na=False)
    & ~df2021["Current Item Name"].str.contains("Grapefruit", case=False, na=False)
    & ~df2021["County"].isin(["State Total", "State Totals", "Sum of Others"])
].copy()

# Counties that have "Grapes, All" — use that directly
has_all = all_grapes2021[all_grapes2021["Current Item Name"] == "Grapes, All"]
counties_with_all = set(has_all["County"])

# Counties without "Grapes, All" — sum the type-level aggregates
# (e.g. "Grapes, Wine, All", "Grapes, Table, All", "Grapes, Raisin, All")
missing = all_grapes2021[
    ~all_grapes2021["County"].isin(counties_with_all)
    & all_grapes2021["Current Item Name"].str.endswith(", All")
]
missing_summed = missing.groupby("County")["Production"].sum().reset_index()

county_prod2021 = pd.concat([
    has_all[["County", "Production"]],
    missing_summed
], ignore_index=True)
county_prod2021.columns = ["county", "total_production_tons"]
county_prod2021 = county_prod2021.dropna(subset=["total_production_tons"])

print(f"Counties from 'Grapes, All': {len(counties_with_all)}")
print(f"Counties from sub-type aggregates: {len(missing_summed)}")
print(county_prod2021[county_prod2021["county"] == "San Diego"])

In [ ]:
# 2022: same approach — use "Grapes, All" where available, sum sub-type
# aggregates for counties that only report by grape type
all_grapes2022 = df2022[
    df2022["Current Item Name"].str.contains("Grapes", case=False, na=False)
    & ~df2022["Current Item Name"].str.contains("Grapefruit", case=False, na=False)
    & ~df2022["County"].isin(["State Total", "State Totals", "Sum of Others"])
].copy()

has_all = all_grapes2022[all_grapes2022["Current Item Name"] == "Grapes, All"]
counties_with_all = set(has_all["County"])

missing = all_grapes2022[
    ~all_grapes2022["County"].isin(counties_with_all)
    & all_grapes2022["Current Item Name"].str.endswith(", All")
]
missing_summed = missing.groupby("County")["Production"].sum().reset_index()

county_prod2022 = pd.concat([
    has_all[["County", "Production"]],
    missing_summed
], ignore_index=True)
county_prod2022.columns = ["county", "total_production_tons"]
county_prod2022 = county_prod2022.dropna(subset=["total_production_tons"])

print(f"Counties from 'Grapes, All': {len(counties_with_all)}")
print(f"Counties from sub-type aggregates: {len(missing_summed)}")
print(county_prod2022[county_prod2022["county"] == "San Diego"])

# Load CA vineyard acreage by county file and year

In [ ]:
# load CA vineyard acreage by county file
vineyards = pd.read_excel("vineyard_acres.xlsx")

In [ ]:
#2014
vineyards2014 = vineyards[vineyards["year"] == 2014]

print(vineyards2014.shape)
print(vineyards2014.head(47))

In [ ]:
#2016
vineyards2016 = vineyards[vineyards["year"] == 2016]

print(vineyards2016.shape)
print(vineyards2016.head(49))

In [ ]:
#2018
vineyards2018 = vineyards[vineyards["year"] == 2018]

print(vineyards2018.shape)
print(vineyards2018.head(49))

In [ ]:
#2019
vineyards2019 = vineyards[vineyards["year"] == 2019]

print(vineyards2019.shape)
print(vineyards2019.head(49))

In [ ]:
#2020
vineyards2020 = vineyards[vineyards["year"] == 2020]

print(vineyards2020.shape)
print(vineyards2020.head(47))

In [ ]:
#2021
vineyards2021 = vineyards[vineyards["year"] == 2021]

print(vineyards2021.shape)
print(vineyards2021.head(49))

In [ ]:
#2022
vineyards2022 = vineyards[vineyards["year"] == 2022]

print(vineyards2022.shape)
print(vineyards2022.head(48))

# Merge production with vineyard acreage and compute yield (tons/acre)

In [ ]:
# Merge total grape production with vineyard acreage from DWR data

yield_acres2014 = county_prod2014.merge(vineyards2014, on="county", how="left")
yield_acres2016 = county_prod2016.merge(vineyards2016, on="county", how="left")
yield_acres2018 = county_prod2018.merge(vineyards2018, on="county", how="left")
yield_acres2019 = county_prod2019.merge(vineyards2019, on="county", how="left")
yield_acres2020 = county_prod2020.merge(vineyards2020, on="county", how="left")
yield_acres2021 = county_prod2021.merge(vineyards2021, on="county", how="left")
yield_acres2022 = county_prod2022.merge(vineyards2022, on="county", how="left")

# Compute yield: total production (tons) / vineyard acreage

In [ ]:
# Yield (tons/acre) = total grape production (tons) / vineyard acreage (DWR)

for df in [yield_acres2014, yield_acres2016, yield_acres2018, yield_acres2019,
           yield_acres2020, yield_acres2021, yield_acres2022]:
    df["yield_per_acre"] = df["total_production_tons"] / df["vineyard_acres"]

# Check 2014 as a sanity check
print("2014 sample:")
print(yield_acres2014[["county", "total_production_tons", "vineyard_acres", "yield_per_acre"]].head(10))

In [ ]:
# Load California county boundaries
counties = gpd.read_file("ca_counties/CA_Counties.shp")

# Identify county column automatically
print(counties.columns)

# Replace with the correct column if different
counties["county"] = counties["NAME"].str.strip().str.title()

In [ ]:
# counties to display
selected_counties = [
    "Napa",
    "Santa Barbara",
    "Fresno",
    "Tehama",
    "Marin",
    "San Diego"
]

# Filter shapefile
counties_subset = counties[counties["county"].isin(selected_counties)]

In [ ]:
# Merge spatial + yield data

#2014
map_df2014 = counties_subset.merge(
    yield_acres2014,
    on="county",
    how="left"
)

#2016
map_df2016 = counties_subset.merge(
    yield_acres2016,
    on="county",
    how="left"
)

#2018
map_df2018 = counties_subset.merge(
    yield_acres2018,
    on="county",
    how="left"
)

#2019
map_df2019 = counties_subset.merge(
    yield_acres2019,
    on="county",
    how="left"
)

#2020
map_df2020 = counties_subset.merge(
    yield_acres2020,
    on="county",
    how="left"
)

#2021
map_df2021 = counties_subset.merge(
    yield_acres2021,
    on="county",
    how="left"
)

#2022
map_df2022 = counties_subset.merge(
    yield_acres2022,
    on="county",
    how="left"
)

# Plot the heatmap

In [ ]:
# 2014
# Plot heatmap
fig, ax = plt.subplots(figsize=(12,12))

# Plot all counties in grey
counties.plot(
    color="lightgrey",
    edgecolor="black",
    linewidth=0.5,
    ax=ax
)

# Plot selected counties with heatmap
map_df2014.plot(
    column="yield_per_acre",
    cmap="Reds",
    linewidth=0.6,
    edgecolor="black",
    legend=True,
    legend_kwds={
        "label": "Grape Yield per Acre",
        "shrink": 0.6
    },
    missing_kwds={
        "color": "lightgrey",
        "label": "No Data"
    },
    ax=ax
)

# Add county labels
for idx, row in map_df2014.iterrows():
    if row["geometry"] is not None:
        centroid = row.geometry.centroid
        ax.text(
            centroid.x,
            centroid.y,
            row["county"],
            fontsize=6,
            ha="center"
        )

# Final styling
ax.set_title(
    "California Grape Yield Heatmap by County (2014)",
    fontsize=18
)

ax.axis("off")

plt.show()

In [ ]:
# 2016
# Plot heatmap
fig, ax = plt.subplots(figsize=(12,12))

# Plot all counties in grey
counties.plot(
    color="lightgrey",
    edgecolor="black",
    linewidth=0.5,
    ax=ax
)

# Plot selected counties with heatmap
map_df2016.plot(
    column="yield_per_acre",
    cmap="Reds",
    linewidth=0.6,
    edgecolor="black",
    legend=True,
    legend_kwds={
        "label": "Grape Yield per Acre",
        "shrink": 0.6
    },
    missing_kwds={
        "color": "lightgrey",
        "label": "No Data"
    },
    ax=ax
)

# Add county labels
for idx, row in map_df2016.iterrows():
    if row["geometry"] is not None:
        centroid = row.geometry.centroid
        ax.text(
            centroid.x,
            centroid.y,
            row["county"],
            fontsize=6,
            ha="center"
        )

# Final styling
ax.set_title(
    "California Grape Yield Heatmap by County (2016)",
    fontsize=18
)

ax.axis("off")

plt.show()

In [ ]:
# 2018
# Plot heatmap
fig, ax = plt.subplots(figsize=(12,12))

# Plot all counties in grey
counties.plot(
    color="lightgrey",
    edgecolor="black",
    linewidth=0.5,
    ax=ax
)

# Plot selected counties with heatmap
map_df2018.plot(
    column="yield_per_acre",
    cmap="Reds",
    linewidth=0.6,
    edgecolor="black",
    legend=True,
    legend_kwds={
        "label": "Grape Yield per Acre",
        "shrink": 0.6
    },
    missing_kwds={
        "color": "lightgrey",
        "label": "No Data"
    },
    ax=ax
)

# Add county labels
for idx, row in map_df2018.iterrows():
    if row["geometry"] is not None:
        centroid = row.geometry.centroid
        ax.text(
            centroid.x,
            centroid.y,
            row["county"],
            fontsize=6,
            ha="center"
        )

# Final styling
ax.set_title(
    "California Grape Yield Heatmap by County (2018)",
    fontsize=18
)

ax.axis("off")

plt.show()

In [ ]:
# 2019
# Plot heatmap
fig, ax = plt.subplots(figsize=(12,12))

# Plot all counties in grey
counties.plot(
    color="lightgrey",
    edgecolor="black",
    linewidth=0.5,
    ax=ax
)

# Plot selected counties with heatmap
map_df2019.plot(
    column="yield_per_acre",
    cmap="Reds",
    linewidth=0.6,
    edgecolor="black",
    legend=True,
    legend_kwds={
        "label": "Grape Yield per Acre",
        "shrink": 0.6
    },
    missing_kwds={
        "color": "lightgrey",
        "label": "No Data"
    },
    ax=ax
)

# Add county labels
for idx, row in map_df2019.iterrows():
    if row["geometry"] is not None:
        centroid = row.geometry.centroid
        ax.text(
            centroid.x,
            centroid.y,
            row["county"],
            fontsize=6,
            ha="center"
        )

# Final styling
ax.set_title(
    "California Grape Yield Heatmap by County (2019)",
    fontsize=18
)

ax.axis("off")

plt.show()

In [ ]:
# 2020
# Plot heatmap
fig, ax = plt.subplots(figsize=(12,12))

# Plot all counties in grey
counties.plot(
    color="lightgrey",
    edgecolor="black",
    linewidth=0.5,
    ax=ax
)

# Plot selected counties with heatmap
map_df2020.plot(
    column="yield_per_acre",
    cmap="Reds",
    linewidth=0.6,
    edgecolor="black",
    legend=True,
    legend_kwds={
        "label": "Grape Yield per Acre",
        "shrink": 0.6
    },
    missing_kwds={
        "color": "lightgrey",
        "label": "No Data"
    },
    ax=ax
)

# Add county labels
for idx, row in map_df2020.iterrows():
    if row["geometry"] is not None:
        centroid = row.geometry.centroid
        ax.text(
            centroid.x,
            centroid.y,
            row["county"],
            fontsize=6,
            ha="center"
        )

# Final styling
ax.set_title(
    "California Grape Yield Heatmap by County (2020)",
    fontsize=18
)

ax.axis("off")

plt.show()

In [ ]:
# 2021
# Plot heatmap
fig, ax = plt.subplots(figsize=(12,12))

# Plot all counties in grey
counties.plot(
    color="lightgrey",
    edgecolor="black",
    linewidth=0.5,
    ax=ax
)

# Plot selected counties with heatmap
map_df2021.plot(
    column="yield_per_acre",
    cmap="Reds",
    linewidth=0.6,
    edgecolor="black",
    legend=True,
    legend_kwds={
        "label": "Grape Yield per Acre",
        "shrink": 0.6
    },
    missing_kwds={
        "color": "lightgrey",
        "label": "No Data"
    },
    ax=ax
)

# Add county labels
for idx, row in map_df2021.iterrows():
    if row["geometry"] is not None:
        centroid = row.geometry.centroid
        ax.text(
            centroid.x,
            centroid.y,
            row["county"],
            fontsize=6,
            ha="center"
        )

# Final styling
ax.set_title(
    "California Grape Yield Heatmap by County (2021)",
    fontsize=18
)

ax.axis("off")

plt.show()

In [ ]:
# 2022
# Plot heatmap
fig, ax = plt.subplots(figsize=(12,12))

# Plot all counties in grey
counties.plot(
    color="lightgrey",
    edgecolor="black",
    linewidth=0.5,
    ax=ax
)

# Plot selected counties with heatmap
map_df2022.plot(
    column="yield_per_acre",
    cmap="Reds",
    linewidth=0.6,
    edgecolor="black",
    legend=True,
    legend_kwds={
        "label": "Grape Yield per Acre",
        "shrink": 0.6
    },
    missing_kwds={
        "color": "lightgrey",
        "label": "No Data"
    },
    ax=ax
)

# Add county labels
for idx, row in map_df2022.iterrows():
    if row["geometry"] is not None:
        centroid = row.geometry.centroid
        ax.text(
            centroid.x,
            centroid.y,
            row["county"],
            fontsize=6,
            ha="center"
        )

# Final styling
ax.set_title(
    "California Grape Yield Heatmap by County (2022)",
    fontsize=18
)

ax.axis("off")

plt.show()

# CSV output

In [ ]:
# Counties we want in the final output
selected_counties = [
    "Napa",
    "Santa Barbara",
    "Fresno",
    "Tehama",
    "Marin",
    "San Diego"
]

# Collect yield tables with year labels
y2014 = yield_acres2014[["county", "yield_per_acre"]].copy()
y2014["year"] = 2014

y2016 = yield_acres2016[["county", "yield_per_acre"]].copy()
y2016["year"] = 2016

y2018 = yield_acres2018[["county", "yield_per_acre"]].copy()
y2018["year"] = 2018

y2019 = yield_acres2019[["county", "yield_per_acre"]].copy()
y2019["year"] = 2019

y2020 = yield_acres2020[["county", "yield_per_acre"]].copy()
y2020["year"] = 2020

y2021 = yield_acres2021[["county", "yield_per_acre"]].copy()
y2021["year"] = 2021

y2022 = yield_acres2022[["county", "yield_per_acre"]].copy()
y2022["year"] = 2022


# Combine all years
all_years = pd.concat([
    y2014, y2016, y2018, y2019, y2020, y2021, y2022
])

# Keep only selected counties
final_df = all_years[all_years["county"].isin(selected_counties)]

# Pivot so each year is a column
final_table = final_df.pivot(
    index="county",
    columns="year",
    values="yield_per_acre"
).reset_index()

# Save to CSV
final_table.to_csv("grape_normalized_yield_selected_counties.csv", index=False)

print(final_table)

In [ ]:
# Side-by-side comparison: 2014 vs 2022 with shared color scale
import matplotlib.colors as mcolors

# Use a shared color range across both years
vmin = min(map_df2014["yield_per_acre"].min(), map_df2022["yield_per_acre"].min())
vmax = max(map_df2014["yield_per_acre"].max(), map_df2022["yield_per_acre"].max())
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
cmap = plt.cm.Reds

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 12))

# --- 2014 (left) ---
counties.plot(color="lightgrey", edgecolor="black", linewidth=0.5, ax=ax1)
map_df2014.plot(
    column="yield_per_acre",
    cmap=cmap,
    norm=norm,
    linewidth=0.6,
    edgecolor="black",
    legend=False,
    missing_kwds={"color": "lightgrey"},
    ax=ax1
)
for idx, row in map_df2014.iterrows():
    if row["geometry"] is not None:
        c = row.geometry.centroid
        ax1.text(c.x, c.y, row["county"], fontsize=6, ha="center")
ax1.set_title("2014", fontsize=18)
ax1.axis("off")

# --- 2022 (right) ---
counties.plot(color="lightgrey", edgecolor="black", linewidth=0.5, ax=ax2)
map_df2022.plot(
    column="yield_per_acre",
    cmap=cmap,
    norm=norm,
    linewidth=0.6,
    edgecolor="black",
    legend=False,
    missing_kwds={"color": "lightgrey"},
    ax=ax2
)
for idx, row in map_df2022.iterrows():
    if row["geometry"] is not None:
        c = row.geometry.centroid
        ax2.text(c.x, c.y, row["county"], fontsize=6, ha="center")
ax2.set_title("2022", fontsize=18)
ax2.axis("off")

# Shared colorbar on the right
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm._A = []
cbar = fig.colorbar(sm, ax=[ax1, ax2], shrink=0.6, location="right")
cbar.set_label("Grape Yield per Acre", fontsize=12)

fig.suptitle("California Grape Yield by County: 2014 vs 2022", fontsize=20, y=0.95)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()